# Hey Sense Feature Engineering + Customer 360

## Fase 2: Feature Engineering y Customer 360

Este notebook **no hace limpieza ni imputacion**. Consume directamente los outputs materializados por Fase 1 en `data_processed/`.

Contrato entre notebooks:

```text
Fase 1 -> data_processed/*.pkl -> Fase 2 -> customer_360
```

La unidad final de analisis es `user_id`.

## 1. Imports y carga de outputs de Fase 1

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 120)

BASE_DIR = Path.cwd()
PROCESSED_DIR = BASE_DIR / "data_processed"

required_files = {
    "conversaciones": PROCESSED_DIR / "conversaciones_clean.pkl",
    "clientes_model": PROCESSED_DIR / "clientes_model.pkl",
    "productos_model": PROCESSED_DIR / "productos_model.pkl",
    "transacciones_model": PROCESSED_DIR / "transacciones_model.pkl",
}

missing_files = [str(path) for path in required_files.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Faltan outputs de Fase 1. Ejecuta primero HeySense_Intelligence_Engine.ipynb. "
        f"Archivos faltantes: {missing_files}"
    )

conversaciones = pd.read_pickle(required_files["conversaciones"])
clientes_model = pd.read_pickle(required_files["clientes_model"])
productos_model = pd.read_pickle(required_files["productos_model"])
transacciones_model = pd.read_pickle(required_files["transacciones_model"])

pd.DataFrame([
    {"dataset": "conversaciones", "rows": len(conversaciones), "columns": conversaciones.shape[1]},
    {"dataset": "clientes_model", "rows": len(clientes_model), "columns": clientes_model.shape[1]},
    {"dataset": "productos_model", "rows": len(productos_model), "columns": productos_model.shape[1]},
    {"dataset": "transacciones_model", "rows": len(transacciones_model), "columns": transacciones_model.shape[1]},
])

,dataset,rows,columns
0,conversaciones,49912,9
1,clientes_model,15025,25
2,productos_model,38909,19
3,transacciones_model,802213,29


## 2. Utilidades para feature engineering

Este bloque define funciones auxiliares usadas por varias familias de features.

- `safe_divide`: evita divisiones entre cero al construir ratios como `ratio_voz_texto` o `ratio_no_procesadas`.
- `minmax_score`: transforma variables numericas a una escala interpretable de 0 a 100.
- `count_regex` y `sum_regex`: permiten extraer senales conversacionales simples a partir de patrones de texto.

Estas funciones no limpian ni imputan datos raw; solo convierten informacion ya procesada en variables modelables.

In [2]:
def safe_divide(numerator, denominator):
    denominator = denominator.replace(0, np.nan) if isinstance(denominator, pd.Series) else denominator
    return (numerator / denominator).replace([np.inf, -np.inf], np.nan).fillna(0)


def minmax_score(series, higher_is_better=True):
    s = pd.to_numeric(series, errors="coerce").fillna(0)
    min_v, max_v = s.min(), s.max()
    if max_v == min_v:
        score = pd.Series(50, index=s.index, dtype="float64")
    else:
        score = 100 * (s - min_v) / (max_v - min_v)
    if not higher_is_better:
        score = 100 - score
    return score.clip(0, 100).round(2)


def count_regex(series, pattern):
    return series.fillna("").str.contains(pattern, regex=True, case=False, na=False).astype(int)


def sum_regex(series, pattern):
    return series.fillna("").str.count(pattern, flags=re.IGNORECASE)

## 3. Features conversacionales por cliente

Objetivo: convertir las conversaciones con Havi en senales accionables por `user_id`.

Esta seccion construye variables que resumen:

- volumen de interaccion con Havi;
- numero de conversaciones unicas;
- canal preferido: texto o voz;
- intensidad de uso del asistente;
- intencion dominante del cliente;
- temas recurrentes;
- senales de frustracion, urgencia y sentimiento.

La logica usa reglas NLP interpretables con expresiones regulares. Para hackathon esto es util porque permite explicar por que un cliente fue clasificado con una intencion como `cargo_no_reconocido`, `credito`, `cancelacion` o `asesor_humano`.

Estas features alimentan directamente los scores de friccion, fraude, churn e intencion comercial.

In [3]:
conversaciones_fe = conversaciones.copy()
conversaciones_fe["texto_usuario"] = conversaciones_fe["input_clean"].fillna(conversaciones_fe["input"].fillna("").str.lower())
conversaciones_fe["texto_havi"] = conversaciones_fe["output_clean"].fillna(conversaciones_fe["output"].fillna("").str.lower())
conversaciones_fe["texto_full"] = (conversaciones_fe["texto_usuario"] + " " + conversaciones_fe["texto_havi"]).str.lower()

intent_patterns = {
    "intent_cargo_no_reconocido": r"cargo no reconocido|no reconozco|no reconocido|no es mi[o\u00edo]|me est[a\u00e1]n cobrando|fraude|robo|aclaraci[o\u00f3]n",
    "intent_consulta_movimiento": r"movimiento|transacci[o\u00f3]n|cargo|cobro|compra|deposito|dep[o\u00f3]sito|transferencia|spei",
    "intent_credito": r"cr[e\u00e9]dito|prestamo|pr[e\u00e9]stamo|mensualidad|financiamiento|auto|carro|hipoteca",
    "intent_promocion": r"promo|promoci[o\u00f3]n|beneficio|descuento|oferta|supercashback|s[u\u00fa]percashback",
    "intent_cashback": r"cashback|recompensa|devoluci[o\u00f3]n|bonificaci[o\u00f3]n",
    "intent_bloqueo_tarjeta": r"bloque|bloquear|bloqueada|tarjeta|pl[a\u00e1]stico|nip|cvv",
    "intent_cancelacion": r"cancelar|cancelaci[o\u00f3]n|cerrar cuenta|dar de baja|baja de cuenta|no quiero",
    "intent_soporte_app": r"app|aplicaci[o\u00f3]n|login|contrase[n\u00f1]a|no puedo entrar|error|falla|no funciona",
    "intent_asesor_humano": r"asesor|agente|ejecutivo|humano|llamar|tel[e\u00e9]fono|contacto|soporte",
}

for col, pattern in intent_patterns.items():
    conversaciones_fe[col] = count_regex(conversaciones_fe["texto_usuario"], pattern)

frustration_pattern = r"problema|error|no puedo|no funciona|rechaz|cancelar|fraude|robo|queja|p[e\u00e9]simo|molesto|no reconozco|no es mi[o\u00edo]"
urgency_pattern = r"urgente|ahora|ya|inmediato|hoy|r[a\u00e1]pido|necesito|ayuda|bloquear|fraude|robo"
positive_pattern = r"gracias|perfecto|excelente|bien|listo|ok|de acuerdo"
negative_pattern = r"problema|error|mal|rechaz|fraude|robo|cancelar|queja|no puedo|no funciona|p[e\u00e9]simo"

conversaciones_fe["frustracion_msg"] = sum_regex(conversaciones_fe["texto_usuario"], frustration_pattern)
conversaciones_fe["urgencia_msg"] = sum_regex(conversaciones_fe["texto_usuario"], urgency_pattern)
conversaciones_fe["sentiment_pos"] = sum_regex(conversaciones_fe["texto_usuario"], positive_pattern)
conversaciones_fe["sentiment_neg"] = sum_regex(conversaciones_fe["texto_usuario"], negative_pattern)
conversaciones_fe["sentimiento_msg"] = conversaciones_fe["sentiment_pos"] - conversaciones_fe["sentiment_neg"]

intent_cols = list(intent_patterns.keys())
conv_basic = conversaciones_fe.groupby("user_id").agg(
    total_interacciones_havi=("input", "count"),
    total_conversaciones=("conv_id", "nunique"),
    total_mensajes_voz=("canal_havi", lambda s: (s == "voz").sum()),
    total_mensajes_texto=("canal_havi", lambda s: (s == "texto").sum()),
    canal_preferido_havi=("canal_havi", lambda s: s.mode().iloc[0] if not s.mode().empty else "desconocido"),
    sentimiento_promedio=("sentimiento_msg", "mean"),
    frustracion_score_raw=("frustracion_msg", "sum"),
    urgencia_score_raw=("urgencia_msg", "sum"),
).reset_index()

conv_intents = conversaciones_fe.groupby("user_id")[intent_cols].sum().reset_index()
conv_features = conv_basic.merge(conv_intents, on="user_id", how="left")
conv_features["ratio_voz_texto"] = safe_divide(conv_features["total_mensajes_voz"], conv_features["total_mensajes_texto"])
conv_features["mensajes_por_conversacion"] = safe_divide(conv_features["total_interacciones_havi"], conv_features["total_conversaciones"])
conv_features["repeticion_problema_score"] = minmax_score(conv_features["mensajes_por_conversacion"] * np.log1p(conv_features["total_conversaciones"]))
conv_features["frustracion_score"] = minmax_score(conv_features["frustracion_score_raw"])
conv_features["urgencia_score"] = minmax_score(conv_features["urgencia_score_raw"])

intent_label_map = {c: c.replace("intent_", "") for c in intent_cols}
conv_features["intencion_dominante"] = conv_features[intent_cols].idxmax(axis=1).map(intent_label_map)
conv_features.loc[conv_features[intent_cols].sum(axis=1).eq(0), "intencion_dominante"] = "sin_intencion_detectada"

def top_intents(row, top_n=3):
    values = row[intent_cols].sort_values(ascending=False)
    selected = [intent_label_map[idx] for idx, val in values.items() if val > 0][:top_n]
    return ", ".join(selected) if selected else "sin_tema_detectado"

conv_features["temas_recurrentes"] = conv_features.apply(top_intents, axis=1)
conv_features = conv_features.drop(columns=["frustracion_score_raw", "urgencia_score_raw"])

print("Features conversacionales:", conv_features.shape)
display(conv_features.head())

Features conversacionales: (15025, 23)


,user_id,total_interacciones_havi,total_conversaciones,total_mensajes_voz,total_mensajes_texto,canal_preferido_havi,sentimiento_promedio,intent_cargo_no_reconocido,intent_consulta_movimiento,intent_credito,intent_promocion,intent_cashback,intent_bloqueo_tarjeta,intent_cancelacion,intent_soporte_app,intent_asesor_humano,ratio_voz_texto,mensajes_por_conversacion,repeticion_problema_score,frustracion_score,urgencia_score,intencion_dominante,temas_recurrentes
0,USR-00001,3,1,0,3,texto,0.0,0,1,2,0,0,2,0,0,0,0.0,3.0,11.1,0.0,0.0,credito,"credito, bloqueo_tarjeta, consulta_movimiento"
1,USR-00002,3,1,0,3,texto,0.0,0,0,2,0,0,0,0,0,1,0.0,3.0,11.1,0.0,8.33,credito,"credito, asesor_humano"
2,USR-00003,3,1,0,3,texto,0.0,0,0,0,0,0,0,0,0,0,0.0,3.0,11.1,0.0,0.0,sin_intencion_detectada,sin_tema_detectado
3,USR-00004,3,1,0,3,texto,-0.666667,0,0,2,0,0,2,2,0,0,0.0,3.0,11.1,22.22,8.33,credito,"credito, bloqueo_tarjeta, cancelacion"
4,USR-00005,5,1,0,5,texto,0.0,0,0,2,0,0,0,0,0,0,0.0,5.0,22.2,0.0,0.0,credito,credito


## 4. Features transaccionales por cliente

Objetivo: resumir el comportamiento financiero real de cada cliente a partir de sus movimientos.

Esta seccion agrega transacciones por `user_id` para crear senales como:

- frecuencia transaccional;
- monto total, promedio, mediano y maximo;
- gasto por categoria MCC;
- operaciones no procesadas;
- disputas y reversos;
- reintentos;
- transacciones internacionales;
- patrones atipicos;
- rechazos por saldo insuficiente o limite excedido;
- uso nocturno y de fin de semana.

Estas variables permiten detectar estres financiero, riesgo operativo, posible fraude, habitos de consumo y oportunidades de recomendacion personalizada.

In [4]:
transacciones_fe = transacciones_model.copy()
transacciones_fe["is_no_procesada"] = transacciones_fe["estatus"].eq("no_procesada").astype(int)
transacciones_fe["is_disputa"] = transacciones_fe["estatus"].eq("en_disputa").astype(int)
transacciones_fe["is_revertida"] = transacciones_fe["estatus"].eq("revertida").astype(int)
transacciones_fe["is_reintento"] = (transacciones_fe["intento_numero"] > 1).astype(int)
transacciones_fe["is_internacional"] = transacciones_fe["es_internacional"].fillna(False).astype(int)
transacciones_fe["is_atipica"] = transacciones_fe["patron_uso_atipico"].fillna(False).astype(int)
transacciones_fe["rechazo_saldo_insuficiente"] = transacciones_fe["motivo_no_procesada"].eq("saldo_insuficiente").astype(int)
transacciones_fe["rechazo_limite_excedido"] = transacciones_fe["motivo_no_procesada"].isin(["limite_excedido", "monto_excede_limite_diario"]).astype(int)
transacciones_fe["is_fin_semana"] = transacciones_fe["dia_semana"].isin(["Saturday", "Sunday"]).astype(int)
transacciones_fe["is_nocturna"] = transacciones_fe["hora_del_dia"].between(0, 5, inclusive="both").astype(int)

category_pivot = (
    transacciones_fe
    .pivot_table(index="user_id", columns="categoria_mcc", values="monto", aggfunc="sum", fill_value=0)
    .add_prefix("gasto_")
    .reset_index()
)

txn_features = transacciones_fe.groupby("user_id").agg(
    total_transacciones=("transaccion_id", "count"),
    monto_total=("monto", "sum"),
    monto_promedio=("monto", "mean"),
    monto_mediano=("monto", "median"),
    monto_max=("monto", "max"),
    num_no_procesadas=("is_no_procesada", "sum"),
    num_disputas=("is_disputa", "sum"),
    num_revertidas=("is_revertida", "sum"),
    num_reintentos=("is_reintento", "sum"),
    num_transacciones_internacionales=("is_internacional", "sum"),
    num_patrones_atipicos=("is_atipica", "sum"),
    rechazos_saldo_insuficiente=("rechazo_saldo_insuficiente", "sum"),
    rechazos_limite_excedido=("rechazo_limite_excedido", "sum"),
    tx_fin_semana=("is_fin_semana", "sum"),
    tx_nocturnas=("is_nocturna", "sum"),
    cashback_total=("cashback_generado", "sum"),
    meses_diferidos_promedio=("meses_diferidos", "mean"),
).reset_index()

txn_features["porcentaje_gasto_fin_semana"] = safe_divide(txn_features["tx_fin_semana"], txn_features["total_transacciones"])
txn_features["porcentaje_gasto_nocturno"] = safe_divide(txn_features["tx_nocturnas"], txn_features["total_transacciones"])
txn_features["ratio_no_procesadas"] = safe_divide(txn_features["num_no_procesadas"], txn_features["total_transacciones"])
txn_features["ratio_disputas"] = safe_divide(txn_features["num_disputas"], txn_features["total_transacciones"])
txn_features["ratio_atipicas"] = safe_divide(txn_features["num_patrones_atipicos"], txn_features["total_transacciones"])
txn_features["ratio_internacional"] = safe_divide(txn_features["num_transacciones_internacionales"], txn_features["total_transacciones"])

txn_features = txn_features.merge(category_pivot, on="user_id", how="left")
for col in ["gasto_delivery", "gasto_restaurante", "gasto_supermercado", "gasto_servicios_digitales"]:
    if col not in txn_features.columns:
        txn_features[col] = 0

print("Features transaccionales:", txn_features.shape)
display(txn_features.head())

Features transaccionales: (15025, 39)


,user_id,total_transacciones,monto_total,monto_promedio,monto_mediano,monto_max,num_no_procesadas,num_disputas,num_revertidas,num_reintentos,num_transacciones_internacionales,num_patrones_atipicos,rechazos_saldo_insuficiente,rechazos_limite_excedido,tx_fin_semana,tx_nocturnas,cashback_total,meses_diferidos_promedio,porcentaje_gasto_fin_semana,porcentaje_gasto_nocturno,ratio_no_procesadas,ratio_disputas,ratio_atipicas,ratio_internacional,gasto_educacion,gasto_entretenimiento,gasto_gobierno,gasto_hogar,gasto_restaurante,gasto_retiro_cajero,gasto_ropa_accesorios,gasto_salud,gasto_servicios_digitales,gasto_supermercado,gasto_tecnologia,gasto_transferencia,gasto_transporte,gasto_viajes,gasto_delivery
0,USR-00001,56,119570.59,2135.189107,437.370,24010.0,3,0,0,2,2,0,1,0,20,15,122.33,0.000000,0.357143,0.267857,0.053571,0.000000,0.0,0.035714,0.0,3038.55,2057.95,0.0,6040.89,0.0,665.36,0.00,6671.82,839.38,0.00,100256.64,0.0,0.00,0
1,USR-00002,74,212722.67,2874.630676,425.615,24830.0,1,1,1,1,1,0,1,0,15,13,147.08,0.000000,0.202703,0.175676,0.013514,0.013514,0.0,0.013514,0.0,2344.75,0.00,0.0,8058.26,0.0,2215.13,0.00,6411.86,1400.74,409.21,191882.72,0.0,0.00,0
2,USR-00003,79,190801.09,2415.203671,559.020,24540.0,4,3,1,1,4,0,2,1,18,17,165.43,0.000000,0.227848,0.215190,0.050633,0.037975,0.0,0.050633,0.0,2872.38,10322.28,0.0,4682.51,0.0,2802.62,0.00,9028.92,554.59,1620.92,158916.87,0.0,0.00,0
3,USR-00004,66,580113.74,8789.602121,3081.140,55000.0,2,2,1,1,2,0,0,0,18,14,0.00,0.545455,0.272727,0.212121,0.030303,0.030303,0.0,0.030303,0.0,6516.53,7642.04,0.0,9710.58,0.0,8878.91,4131.28,2195.00,15842.70,6893.13,508000.00,0.0,10303.57,0
4,USR-00005,93,190277.30,2045.992473,393.990,24520.0,2,2,0,2,1,0,0,0,27,27,162.99,0.000000,0.290323,0.290323,0.021505,0.021505,0.0,0.010753,0.0,5899.38,471.22,0.0,6310.69,0.0,976.62,0.00,6955.11,1214.96,1106.79,167342.53,0.0,0.00,0


## 5. Features de producto por cliente

Objetivo: representar el portafolio financiero de cada cliente en una sola fila.

Esta seccion convierte la tabla de productos en senales como:

- si tiene tarjeta de credito;
- si tiene inversion;
- si tiene seguro;
- si tiene credito personal o automotriz;
- numero de productos activos;
- utilizacion maxima y promedio del credito;
- limite de credito total;
- saldo total;
- mensualidad total de creditos;
- tasa promedio.

Estas features son clave para saber que productos ya tiene el cliente, que tan comprometida esta su capacidad financiera y que recomendaciones serian razonables o inoportunas.

In [5]:
productos_fe = productos_model.copy()
productos_fe["is_activo"] = productos_fe["estatus"].eq("activo").astype(int)
productos_fe["is_tarjeta_credito"] = productos_fe["tipo_producto"].str.contains("tarjeta_credito", na=False).astype(int)
productos_fe["is_inversion"] = productos_fe["tipo_producto"].eq("inversion_hey").astype(int)
productos_fe["is_seguro"] = productos_fe["tipo_producto"].isin(["seguro_vida", "seguro_compras"]).astype(int)
productos_fe["is_credito_personal"] = productos_fe["tipo_producto"].eq("credito_personal").astype(int)
productos_fe["is_credito_auto"] = productos_fe["tipo_producto"].eq("credito_auto").astype(int)

prod_features = productos_fe.groupby("user_id").agg(
    tiene_tarjeta_credito=("is_tarjeta_credito", "max"),
    tiene_inversion=("is_inversion", "max"),
    tiene_seguro_producto=("is_seguro", "max"),
    tiene_credito_personal=("is_credito_personal", "max"),
    tiene_credito_auto=("is_credito_auto", "max"),
    num_productos=("producto_id", "count"),
    num_productos_activos_calc=("is_activo", "sum"),
    utilizacion_credito_max=("utilizacion_pct", "max"),
    utilizacion_credito_promedio=("utilizacion_pct", "mean"),
    limite_credito_total=("limite_credito", "sum"),
    saldo_total=("saldo_actual", "sum"),
    mensualidad_total_creditos=("monto_mensualidad", "sum"),
    tasa_interes_promedio=("tasa_interes_anual", "mean"),
).reset_index()

print("Features de producto:", prod_features.shape)
display(prod_features.head())

Features de producto: (15025, 14)


,user_id,tiene_tarjeta_credito,tiene_inversion,tiene_seguro_producto,tiene_credito_personal,tiene_credito_auto,num_productos,num_productos_activos_calc,utilizacion_credito_max,utilizacion_credito_promedio,limite_credito_total,saldo_total,mensualidad_total_creditos,tasa_interes_promedio
0,USR-00001,1,0,0,0,0,2,2,0.6166,0.308300,144000.0,169745.00,0.0,17.855
1,USR-00002,1,0,0,0,0,2,2,0.2783,0.139150,22000.0,26835.14,0.0,17.040
2,USR-00003,1,0,0,0,0,2,2,0.3091,0.154550,5000.0,5000.15,0.0,19.940
3,USR-00004,1,1,0,0,0,3,3,0.3538,0.117933,67000.0,357230.61,0.0,15.630
4,USR-00005,1,0,0,0,0,2,2,0.6326,0.316300,136000.0,129359.71,0.0,15.475


## 6. Customer 360 Table

Objetivo: construir una vista unificada de cliente.

`customer_360` combina en una sola tabla:

- datos demograficos y bancarios de `clientes_model`;
- senales conversacionales de `conv_features`;
- comportamiento financiero de `txn_features`;
- portafolio de productos de `prod_features`.

La tabla resultante tiene una fila por `user_id` y se convierte en la base para modelos, segmentacion, Need Detection Engine, Next Best Action y dashboard.

Despues del merge, solo se rellenan nulos generados por ausencia de actividad en alguna fuente, por ejemplo clientes sin conversacion o sin cierto tipo de transaccion. Esto no reemplaza la limpieza de Fase 1.

In [6]:
customer_360 = clientes_model.merge(conv_features, on="user_id", how="left")
customer_360 = customer_360.merge(txn_features, on="user_id", how="left")
customer_360 = customer_360.merge(prod_features, on="user_id", how="left")

numeric_cols_c360 = customer_360.select_dtypes(include=["number", "boolean"]).columns
customer_360[numeric_cols_c360] = customer_360[numeric_cols_c360].fillna(0)

for col, default in {
    "canal_preferido_havi": "sin_interaccion_havi",
    "intencion_dominante": "sin_interaccion_havi",
    "temas_recurrentes": "sin_interaccion_havi",
}.items():
    if col in customer_360.columns:
        customer_360[col] = customer_360[col].fillna(default)

print("Customer 360:", customer_360.shape)
print("Usuarios unicos:", customer_360["user_id"].nunique())
display(customer_360.head())

Customer 360: (15025, 98)
Usuarios unicos: 15025


,user_id,edad,sexo,estado,ciudad,nivel_educativo,ocupacion,ingreso_mensual_mxn,antiguedad_dias,es_hey_pro,nomina_domiciliada,canal_apertura,score_buro,dias_desde_ultimo_login,preferencia_canal,satisfaccion_1_10,recibe_remesas,usa_hey_shop,idioma_preferido,tiene_seguro,num_productos_activos,patron_uso_atipico,satisfaccion_1_10_was_missing,estado_was_missing,ciudad_was_missing,total_interacciones_havi,total_conversaciones,total_mensajes_voz,total_mensajes_texto,canal_preferido_havi,sentimiento_promedio,intent_cargo_no_reconocido,intent_consulta_movimiento,intent_credito,intent_promocion,intent_cashback,intent_bloqueo_tarjeta,intent_cancelacion,intent_soporte_app,intent_asesor_humano,ratio_voz_texto,mensajes_por_conversacion,repeticion_problema_score,frustracion_score,urgencia_score,intencion_dominante,temas_recurrentes,total_transacciones,monto_total,monto_promedio,monto_mediano,monto_max,num_no_procesadas,num_disputas,num_revertidas,num_reintentos,num_transacciones_internacionales,num_patrones_atipicos,rechazos_saldo_insuficiente,rechazos_limite_excedido,tx_fin_semana,tx_nocturnas,cashback_total,meses_diferidos_promedio,porcentaje_gasto_fin_semana,porcentaje_gasto_nocturno,ratio_no_procesadas,ratio_disputas,ratio_atipicas,ratio_internacional,gasto_educacion,gasto_entretenimiento,gasto_gobierno,gasto_hogar,gasto_restaurante,gasto_retiro_cajero,gasto_ropa_accesorios,gasto_salud,gasto_servicios_digitales,gasto_supermercado,gasto_tecnologia,gasto_transferencia,gasto_transporte,gasto_viajes,gasto_delivery,tiene_tarjeta_credito,tiene_inversion,tiene_seguro_producto,tiene_credito_personal,tiene_credito_auto,num_productos,num_productos_activos_calc,utilizacion_credito_max,utilizacion_credito_promedio,limite_credito_total,saldo_total,mensualidad_total_creditos,tasa_interes_promedio
0,USR-00001,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,False,App,527,1,app_android,10.0,False,True,es_MX,False,2,False,False,False,False,3,1,0,3,texto,0.0,0,1,2,0,0,2,0,0,0,0.0,3.0,11.1,0.0,0.0,credito,"credito, bloqueo_tarjeta, consulta_movimiento",56,119570.59,2135.189107,437.370,24010.0,3,0,0,2,2,0,1,0,20,15,122.33,0.000000,0.357143,0.267857,0.053571,0.000000,0.0,0.035714,0.0,3038.55,2057.95,0.0,6040.89,0.0,665.36,0.00,6671.82,839.38,0.00,100256.64,0.0,0.00,0,1,0,0,0,0,2,2,0.6166,0.308300,144000.0,169745.00,0.0,17.855
1,USR-00002,18,M,Jalisco,Puerto Vallarta,Preparatoria,Estudiante,19000,1410,True,False,App,714,3,app_android,8.0,False,True,es_MX,True,2,False,False,False,False,3,1,0,3,texto,0.0,0,0,2,0,0,0,0,0,1,0.0,3.0,11.1,0.0,8.33,credito,"credito, asesor_humano",74,212722.67,2874.630676,425.615,24830.0,1,1,1,1,1,0,1,0,15,13,147.08,0.000000,0.202703,0.175676,0.013514,0.013514,0.0,0.013514,0.0,2344.75,0.00,0.0,8058.26,0.0,2215.13,0.00,6411.86,1400.74,409.21,191882.72,0.0,0.00,0,1,0,0,0,0,2,2,0.2783,0.139150,22000.0,26835.14,0.0,17.040
2,USR-00003,23,H,Chihuahua,Cuauhtémoc,Licenciatura,Estudiante,14000,1174,True,False,App,454,3,app_ios,8.0,False,True,es_MX,False,2,False,False,False,False,3,1,0,3,texto,0.0,0,0,0,0,0,0,0,0,0,0.0,3.0,11.1,0.0,0.0,sin_intencion_detectada,sin_tema_detectado,79,190801.09,2415.203671,559.020,24540.0,4,3,1,1,4,0,2,1,18,17,165.43,0.000000,0.227848,0.215190,0.050633,0.037975,0.0,0.050633,0.0,2872.38,10322.28,0.0,4682.51,0.0,2802.62,0.00,9028.92,554.59,1620.92,158916.87,0.0,0.00,0,1,0,0,0,0,2,2,0.3091,0.154550,5000.0,5000.15,0.0,19.940
3,USR-00004,32,SE,Nuevo León,Guadalupe,Posgrado,Empleado,61000,1168,False,False,Fan Shop,837,16,app_ios,10.0,True,False,es_MX,True,3,False,False,False,False,3,1,0,3,texto,-0.666667,0,0,2,0,0,2,2,0,0,0.0,3.0,11.1,22.22,8.33,credito,"credito, bloqueo_tarjeta, cancelacion",66,580113.74,8789.602121,3081.140,55000.0,2,2,1,1,2,0,0,0,18,14,0.00,0.545455,0.272727,0.212121,0.030303,0.030303,0.0,0.030303,0.0,6516.53,7642.04,0.0,9710.58,0.0,8878.91,4131.28,2195.00,15842.70,6893.13,508000.00,0.0,10303.57,0,1,1,0,0,0,3,3,0.3538,0.117933,67000.0,357230.61,0.0,15.630
4,USR-00005,

## 7. Features financieras derivadas

Objetivo: transformar muchas variables individuales en scores interpretables de negocio.

Los scores se escalan de 0 a 100 para facilitar lectura en dashboard y storytelling:

- `liquidez_score`: capacidad general basada en ingreso, saldo y baja friccion de pagos.
- `estres_financiero_score`: presion financiera por rechazos, gasto variable, utilizacion de credito y frustracion.
- `capacidad_ahorro_score`: potencial de ahorro o inversion.
- `riesgo_friccion_score`: senales de mala experiencia o necesidad de atencion prioritaria.
- `riesgo_fraude_score`: actividad atipica, disputas, internacionalidad y menciones de cargos no reconocidos.
- `propension_credito_score`: probabilidad razonable de interes o elegibilidad para credito.
- `propension_inversion_score`: oportunidad de recomendar inversion o ahorro automatizado.
- `propension_seguro_score`: oportunidad de proteccion financiera.
- `churn_risk_score`: riesgo de abandono o caida de engagement.

Estos scores no son modelos predictivos entrenados todavia; son features derivadas explicables que sirven como baseline para reglas, segmentacion y priorizacion de acciones.

In [7]:
income_score = minmax_score(customer_360["ingreso_mensual_mxn"])
saldo_score = minmax_score(customer_360["saldo_total"])
low_rejection_score = minmax_score(customer_360["rechazos_saldo_insuficiente"] + customer_360["num_no_procesadas"], higher_is_better=False)

customer_360["liquidez_score"] = (0.45 * income_score + 0.35 * saldo_score + 0.20 * low_rejection_score).round(2).clip(0, 100)
customer_360["estres_financiero_score"] = (
    0.30 * minmax_score(customer_360["rechazos_saldo_insuficiente"] + customer_360["num_no_procesadas"]) +
    0.25 * minmax_score(customer_360["gasto_delivery"] + customer_360["gasto_restaurante"] + customer_360["gasto_servicios_digitales"]) +
    0.20 * minmax_score(customer_360["utilizacion_credito_max"]) +
    0.15 * minmax_score(customer_360["mensualidad_total_creditos"]) +
    0.10 * minmax_score(customer_360["frustracion_score"])
).round(2).clip(0, 100)
customer_360["capacidad_ahorro_score"] = (
    0.35 * income_score + 0.25 * saldo_score + 0.20 * minmax_score(customer_360["satisfaccion_1_10"]) + 0.20 * minmax_score(customer_360["nomina_domiciliada"].astype(int))
).round(2).clip(0, 100)
customer_360["riesgo_friccion_score"] = (
    0.30 * minmax_score(customer_360["frustracion_score"]) + 0.20 * minmax_score(customer_360["num_no_procesadas"]) + 0.20 * minmax_score(customer_360["num_disputas"]) + 0.15 * minmax_score(customer_360["intent_asesor_humano"]) + 0.15 * minmax_score(customer_360["satisfaccion_1_10"], higher_is_better=False)
).round(2).clip(0, 100)
customer_360["riesgo_fraude_score"] = (
    0.35 * minmax_score(customer_360["num_patrones_atipicos"]) + 0.25 * minmax_score(customer_360["num_disputas"]) + 0.20 * minmax_score(customer_360["num_transacciones_internacionales"]) + 0.20 * minmax_score(customer_360["intent_cargo_no_reconocido"])
).round(2).clip(0, 100)
customer_360["propension_credito_score"] = (
    0.30 * minmax_score(customer_360["intent_credito"]) + 0.25 * minmax_score(customer_360["score_buro"]) + 0.20 * income_score + 0.15 * minmax_score(customer_360["nomina_domiciliada"].astype(int)) + 0.10 * minmax_score(1 - customer_360["tiene_credito_personal"].astype(int))
).round(2).clip(0, 100)
customer_360["propension_inversion_score"] = (
    0.30 * customer_360["capacidad_ahorro_score"] + 0.25 * minmax_score(1 - customer_360["tiene_inversion"].astype(int)) + 0.20 * minmax_score(customer_360["satisfaccion_1_10"]) + 0.15 * minmax_score(customer_360["monto_total"]) + 0.10 * minmax_score(customer_360["antiguedad_dias"])
).round(2).clip(0, 100)
customer_360["propension_seguro_score"] = (
    0.30 * minmax_score(1 - customer_360["tiene_seguro"].astype(int)) + 0.25 * minmax_score(customer_360["edad"]) + 0.20 * income_score + 0.15 * minmax_score(customer_360["num_productos_activos"].fillna(customer_360["num_productos_activos_calc"])) + 0.10 * minmax_score(customer_360["satisfaccion_1_10"])
).round(2).clip(0, 100)
customer_360["churn_risk_score"] = (
    0.25 * customer_360["riesgo_friccion_score"] + 0.20 * minmax_score(customer_360["dias_desde_ultimo_login"]) + 0.20 * minmax_score(customer_360["satisfaccion_1_10"], higher_is_better=False) + 0.15 * minmax_score(customer_360["intent_cancelacion"]) + 0.10 * minmax_score(customer_360["num_no_procesadas"]) + 0.10 * minmax_score(customer_360["repeticion_problema_score"])
).round(2).clip(0, 100)

score_cols = ["liquidez_score", "estres_financiero_score", "capacidad_ahorro_score", "riesgo_friccion_score", "riesgo_fraude_score", "propension_credito_score", "propension_inversion_score", "propension_seguro_score", "churn_risk_score"]
customer_360[["user_id"] + score_cols].head()

,user_id,liquidez_score,estres_financiero_score,capacidad_ahorro_score,riesgo_friccion_score,riesgo_fraude_score,propension_credito_score,propension_inversion_score,propension_seguro_score,churn_risk_score
0,USR-00001,32.84,21.44,33.59,4.29,3.08,28.68,64.75,52.02,4.44
1,USR-00002,25.60,10.82,20.62,11.21,4.66,35.94,55.28,16.21,10.67
2,USR-00003,16.77,19.37,17.98,17.5,15.53,19.17,52.98,48.13,14.39
3,USR-00004,62.46,14.33,53.94,14.52,9.33,50.37,48.32,39.29,11.28
4,USR-00005,34.64,17.81,24.47,14.29,7.79,29.48,50.09,21.24,15.91


## 8. Validacion de Customer 360

Objetivo: asegurar que la tabla final sea apta para modelado y analisis.

Validaciones incluidas:

- una sola fila por cliente;
- mismo numero de clientes que la base original;
- ausencia de duplicados en `user_id`;
- ausencia de nulos en scores derivados.

Estas validaciones son importantes porque `customer_360` sera la entrada contractual para las siguientes fases del motor.

In [8]:
customer_360_checks = pd.DataFrame([
    {"check": "Una fila por cliente", "valor": int(customer_360["user_id"].is_unique), "status": "OK" if customer_360["user_id"].is_unique else "REVISAR"},
    {"check": "Total clientes", "valor": int(customer_360["user_id"].nunique()), "status": "INFO"},
    {"check": "Filas customer_360", "valor": int(len(customer_360)), "status": "INFO"},
    {"check": "Duplicados user_id", "valor": int(customer_360["user_id"].duplicated().sum()), "status": "OK" if customer_360["user_id"].duplicated().sum() == 0 else "REVISAR"},
    {"check": "Nulos en scores derivados", "valor": int(customer_360[score_cols].isna().sum().sum()), "status": "OK" if customer_360[score_cols].isna().sum().sum() == 0 else "REVISAR"},
])
customer_360_checks

,check,valor,status
0,Una fila por cliente,1,OK
1,Total clientes,15025,INFO
2,Filas customer_360,15025,INFO
3,Duplicados user_id,0,OK
4,Nulos en scores derivados,0,OK


## 9. Vista ejecutiva de clientes accionables

Objetivo: identificar clientes reales del dataset que sirvan para demo, storytelling y validacion cualitativa.

La tabla `clientes_demo` prioriza clientes con senales fuertes de:

- riesgo de fraude;
- friccion;
- estres financiero;
- interacciones relevantes con Havi;
- patrones atipicos o disputas.

Esta vista ayuda a seleccionar casos concretos para mostrar como Hey Sense detecta necesidades implicitas y propone acciones personalizadas.

In [9]:
demo_columns = [
    "user_id", "edad", "ocupacion", "ingreso_mensual_mxn", "satisfaccion_1_10", "intencion_dominante", "temas_recurrentes",
    "total_transacciones", "num_no_procesadas", "num_disputas", "num_patrones_atipicos",
    "liquidez_score", "estres_financiero_score", "riesgo_friccion_score", "riesgo_fraude_score", "propension_inversion_score", "churn_risk_score",
]
clientes_demo = customer_360.sort_values(["riesgo_fraude_score", "riesgo_friccion_score", "estres_financiero_score"], ascending=False)[demo_columns].head(15)
clientes_demo

,user_id,edad,ocupacion,ingreso_mensual_mxn,satisfaccion_1_10,intencion_dominante,temas_recurrentes,total_transacciones,num_no_procesadas,num_disputas,num_patrones_atipicos,liquidez_score,estres_financiero_score,riesgo_friccion_score,riesgo_fraude_score,propension_inversion_score,churn_risk_score
2799,USR-02800,31,Empleado,17000,10.0,soporte_app,soporte_app,90,4,6,90,24.25,20.23,20.71,64.10,56.92,8.47
7931,USR-07932,33,Empresario,85000,7.0,cargo_no_reconocido,"cargo_no_reconocido, consulta_movimiento",100,0,5,100,65.37,2.83,18.93,60.76,60.79,13.97
8830,USR-08831,33,Empleado,24500,7.0,sin_intencion_detectada,sin_tema_detectado,78,1,7,78,29.92,8.52,25.36,58.40,58.08,15.85
8048,USR-08049,18,Estudiante,26500,9.0,credito,credito,93,3,5,93,30.02,17.85,18.93,57.40,55.61,11.73
13806,USR-13807,20,Estudiante,12500,10.0,soporte_app,soporte_app,87,0,5,87,30.59,16.57,15.83,55.31,57.61,5.5
8354,USR-08355,27,Independiente,21000,9.0,credito,credito,95,1,5,95,31.28,11.87,16.07,55.03,53.85,7.7
5338,USR-05339,32,Empleado,27000,8.0,credito,credito,86,2,6,86,30.64,19.95,22.14,55.00,63.75,12.68
2363,USR-02364,28,Empresario,30000,10.0,credito,credito,76,3,7,76,29.01,10.28,21.79,54.63,64.16,8.7
4005,USR-04006,37,Independiente,87000,10.0,consulta_movimiento,"consulta_movimiento, asesor_humano",99,2,3,99,61.50,12.4,13.36,53.26,75.98,5.54
6423,USR-06424,28,Empresario,30000,8.0,sin_intencion_detectada,sin_tema_detectado,95,1,5,95,41.22,20.04,18.21,51.95,65.46,11.75


## 10. Exportar output de Fase 2

Se materializa el resultado de feature engineering para que las siguientes fases no recalculen la tabla completa.

Outputs generados en `data_features/`:

- `customer_360.pkl`: tabla principal por cliente.
- `conv_features.pkl`: features conversacionales.
- `txn_features.pkl`: features transaccionales.
- `prod_features.pkl`: features de producto.
- `clientes_demo.pkl`: clientes priorizados para demo.

Esto crea un contrato claro:

```text
Fase 1 -> data_processed/ -> Fase 2 -> data_features/ -> Fase 3
```

In [10]:
FEATURE_DIR = BASE_DIR / "data_features"
FEATURE_DIR.mkdir(exist_ok=True)

customer_360.to_pickle(FEATURE_DIR / "customer_360.pkl")
conv_features.to_pickle(FEATURE_DIR / "conv_features.pkl")
txn_features.to_pickle(FEATURE_DIR / "txn_features.pkl")
prod_features.to_pickle(FEATURE_DIR / "prod_features.pkl")
clientes_demo.to_pickle(FEATURE_DIR / "clientes_demo.pkl")

print(f"customer_360 guardado en: {FEATURE_DIR / 'customer_360.pkl'} -> {customer_360.shape}")

customer_360 guardado en: c:\Users\carol\OneDrive\Imágenes\Escritorio\Datathon\data_features\customer_360.pkl -> (15025, 107)


## 11. Resultado de Fase 2

La tabla `customer_360` queda lista para:

- segmentacion no supervisada;
- Need Detection Engine;
- Next Best Action Engine;
- dashboard ejecutivo;
- demo de Havi proactivo.

Siguiente fase recomendada: construir segmentos y reglas de necesidades implicitas.